In [0]:
from pyspark.sql.functions import *
import random

random.seed(42)
 
# Listening events (large table - 500K rows)
events_data = []
for i in range(500000):
    events_data.append((
        f"EVT-{i+1:07d}",
        f"USR-{random.randint(1, 100000):06d}",
        f"TRK-{random.randint(1, 50000):06d}",
        f"ART-{random.randint(1, 5000):05d}",
        random.randint(10, 300),
        random.choice([True, False]),
        random.choice(["mobile", "desktop", "smart_speaker", "tablet"]),
        random.choice(["free", "premium"]),
        f"202{random.randint(3,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    ))
 
events = spark.createDataFrame(events_data,
    ["event_id", "user_id", "track_id", "artist_id", "duration_sec",
     "completed", "device", "tier", "event_date"]) \
    .withColumn("event_date", col("event_date").cast("date")) \
    .withColumn("year", year(col("event_date"))) \
    .withColumn("month", month(col("event_date")))
 
events.write.parquet("/Volumes/workspace/default/demo/events", mode="overwrite", partitionBy=["year"])
 
# Artists (small table - 5K rows)
artist_data = [(f"ART-{i+1:05d}", f"Artist {i+1}",
                random.choice(["Pop", "Rock", "Hip-Hop", "Jazz", "Electronic"]),
                random.choice(["US", "UK", "KR", "JP", "DE"]))
               for i in range(5000)]
artists = spark.createDataFrame(artist_data, ["artist_id", "name", "genre", "country"])
artists.write.parquet("/Volumes/workspace/default/demo/artists", mode="overwrite")
 
# Tracks (medium table - 50K rows)
track_data = [(f"TRK-{i+1:06d}", f"Track {i+1}",
               f"ART-{random.randint(1, 5000):05d}",
               random.randint(60, 400),
               random.randint(2018, 2024))
              for i in range(50000)]
tracks = spark.createDataFrame(track_data,
    ["track_id", "title", "artist_id", "track_duration", "release_year"])
tracks.write.parquet("/Volumes/workspace/default/demo/tracks", mode="overwrite")
 
# Reload from Parquet
events = spark.read.parquet("/Volumes/workspace/default/demo/events")
artists = spark.read.parquet("/Volumes/workspace/default/demo/artists")
tracks = spark.read.parquet("/Volumes/workspace/default/demo/tracks")
 
print(f"Events: {events.count()} | Artists: {artists.count()} | Tracks: {tracks.count()}")


Events: 500000 | Artists: 5000 | Tracks: 50000


In [0]:
#Query 1: Simple filter and select
q1 = events.filter(col("year") == 2024) \
    .filter(col("completed") == True) \
    .select("event_id", "user_id", "duration_sec")
 
print("QUERY 1: Simple filter and select")
q1.explain(mode="formatted")

QUERY 1: Simple filter and select
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [5]: [event_id#13381, user_id#13382, duration_sec#13385L, completed#13386, year#13391]
DictionaryFilters: [completed#13386]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/demo/events]
PartitionFilters: [isnotnull(year#13391), (year#13391 = 2024)]
ReadSchema: struct<event_id:string,user_id:string,duration_sec:bigint,completed:boolean>
RequiredDataFilters: [completed#13386, isnotnull(completed#13386)]

(2) PhotonProject
Input [5]: [event_id#13381, user_id#13382, duration_sec#13385L, completed#13386, year#13391]
Arguments: [event_id#13381, user_id#13382, duration_sec#13385L]

(3) PhotonColumnarToRow
Input [3]: [event_id#13381, user_id#13382, duration_sec#13385L]

(4) PhotonResultStage
Input [3]: [event_id#13381, user_id#13382, duration_sec#13385L]


== Photon Explanation ==
The query i

| Aspect | Your Finding |
|---|---|
| Scan type | `PhotonScan parquet` (Parquet scan executed with Photon engine) |
| PartitionFilters | `isnotnull(year)` and `year = 2024` |
| PushedFilters | `completed`, `isnotnull(completed)` |
| ReadSchema columns | `event_id`, `user_id`, `duration_sec`, `completed` |
| Exchange count | `0` (no `Exchange` operator in the plan) |
| Assessment | Efficient plan with partition pruning, predicate pushdown, column pruning, and full Photon support. No shuffle/exchange detected. |

In [0]:
#Query 2: Join events with artists
q2 = events.join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .select("event_id", "name", "genre", "duration_sec")
 
print("QUERY 2: Events JOIN Artists (filter after join)")
q2.explain(mode="formatted")

QUERY 2: Events JOIN Artists (filter after join)
== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin Inner (7)
            :- PhotonProject (2)
            :  +- PhotonScan parquet  (1)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonScan parquet  (3)


(1) PhotonScan parquet 
Output [4]: [event_id#13381, artist_id#13384, duration_sec#13385L, year#13391]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/demo/events]
OptionalDataFilters: [hashedrelationcontains(artist_id#13384)]
PartitionFilters: [isnotnull(year#13391), (year#13391 = 2024)]
ReadSchema: struct<event_id:string,artist_id:string,duration_sec:bigint>
RequiredDataFilters: [isnotnull(artist_id#13384)]

(2) PhotonProject
Input [4]: [event_id#13381, artist_id#13384, 

| Aspect | Your Finding |
|---|---|
| Join strategy | `PhotonBroadcastHashJoin` (Inner Join) |
| Artists table size | `~5K rows (small!)` |
| Exchange count | `1` (`PhotonShuffleExchangeSink` → `PhotonShuffleExchangeSource` for broadcast preparation) |
| Could broadcast? | Yes — the small `artists` table is broadcast to all executors |
| Filter placement | `year = 2024` filter is applied before the join via `PartitionFilters` on the `events` scan |
| Assessment | Efficient join plan. Small dimension table (`artists`) is broadcast, avoiding large shuffle on the `events` table. Partition pruning is applied on `events`, and the query is fully Photon-optimized. |

In [0]:
#Query 3: Three-table join
q3 = events.join(tracks, "track_id") \
    .join(artists, "artist_id") \
    .filter(col("year") == 2024) \
    .filter(col("genre") == "Pop") \
    .groupBy("name") \
    .agg(count("*").alias("play_count"), avg("duration_sec").alias("avg_duration"))
 
print("QUERY 3: Three-table join with aggregation")
q3.explain(mode="formatted")

QUERY 3: Three-table join with aggregation
== Physical Plan ==
AdaptiveSparkPlan (23)
+- == Initial Plan ==
   PhotonResultStage (22)
   +- PhotonColumnarToRow (21)
      +- PhotonGroupingAgg (20)
         +- PhotonShuffleExchangeSource (19)
            +- PhotonShuffleMapStage (18)
               +- PhotonShuffleExchangeSink (17)
                  +- PhotonGroupingAgg (16)
                     +- PhotonProject (15)
                        +- PhotonBroadcastHashJoin Inner (14)
                           :- PhotonProject (8)
                           :  +- PhotonBroadcastHashJoin Inner (7)
                           :     :- PhotonProject (2)
                           :     :  +- PhotonScan parquet  (1)
                           :     +- PhotonShuffleExchangeSource (6)
                           :        +- PhotonShuffleMapStage (5)
                           :           +- PhotonShuffleExchangeSink (4)
                           :              +- PhotonScan parquet  (3)
            

| Aspect | Your Finding |
|---|---|
| Join 1 strategy | `PhotonBroadcastHashJoin` between `events` and `tracks` on `track_id` |
| Join 2 strategy | `PhotonBroadcastHashJoin` between intermediate result and `artists` on `artist_id` |
| Total Exchange count | `3` total exchanges: 2 for broadcast preparation (`tracks`, `artists`) and 1 shuffle for aggregation (`hashpartitioning(name, 16)`) |
| Filter on year? | Yes — pushed to partition pruning via `PartitionFilters: year = 2024` |
| Filter on genre? | Yes — pushed down to scan via `DictionaryFilters` and `RequiredDataFilters: genre = Pop` |
| Assessment | Well-optimized multi-join aggregation query. Both small dimension tables are broadcast, minimizing shuffle during joins. Partition pruning on `year` and predicate pushdown on `genre` reduce scan cost. Only aggregation requires a shuffle, which is expected for grouped results. Fully Photon-supported. |

In [0]:
#Query 4: Aggregation with multiple actions (simulated)
enriched = events.join(artists, "artist_id").filter(col("year") == 2024)
 
print("QUERY 4a: Genre aggregation")
q4a = enriched.groupBy("genre").agg(count("*").alias("plays"))
q4a.explain(mode="formatted")
 
print("\nQUERY 4b: Device aggregation (same enriched source)")
q4b = enriched.groupBy("device").agg(avg("duration_sec").alias("avg_dur"))
q4b.explain(mode="formatted")


QUERY 4a: Genre aggregation
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonProject (8)
                        +- PhotonBroadcastHashJoin Inner (7)
                           :- PhotonProject (2)
                           :  +- PhotonScan parquet  (1)
                           +- PhotonShuffleExchangeSource (6)
                              +- PhotonShuffleMapStage (5)
                                 +- PhotonShuffleExchangeSink (4)
                                    +- PhotonScan parquet  (3)


(1) PhotonScan parquet 
Output [2]: [artist_id#13384, year#13391]
Location: InMemoryFileIndex [dbfs:/Volumes/workspace/default/demo/events]
OptionalDataFilters: [hashedrelationcon

| Aspect | Your Finding |
|---|---|
| Does 4a and 4b share computation? | No — each query independently scans `events` and `artists`, performs the join, and executes its own aggregation |
| Is enriched cached? | No (cache operators are absent) |
| Redundant work | The `events` ↔ `artists` broadcast join is repeated in both queries, causing duplicate scans and repeated join computation |
| Assessment | Both queries are individually efficient and fully Photon-optimized, but together they repeat the same work. If these aggregations are frequently executed together, caching could reduce repeated scans and joins. |

In [0]:
#Query 5: Self-join pattern
popular = events.groupBy("track_id").agg(count("*").alias("play_count")) \
    .filter(col("play_count") > 10)
 
q5 = events.join(popular, "track_id") \
    .select("event_id", "user_id", "track_id", "play_count")
 
print("QUERY 5: Self-reference (events aggregated then joined back)")
q5.explain(mode="formatted")

QUERY 5: Self-reference (events aggregated then joined back)
== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Initial Plan ==
   PhotonResultStage (17)
   +- PhotonColumnarToRow (16)
      +- PhotonProject (15)
         +- PhotonBroadcastHashJoin Inner (14)
            :- PhotonProject (2)
            :  +- PhotonScan parquet  (1)
            +- PhotonShuffleExchangeSource (13)
               +- PhotonShuffleMapStage (12)
                  +- PhotonShuffleExchangeSink (11)
                     +- PhotonFilter (10)
                        +- PhotonGroupingAgg (9)
                           +- PhotonShuffleExchangeSource (8)
                              +- PhotonShuffleMapStage (7)
                                 +- PhotonShuffleExchangeSink (6)
                                    +- PhotonGroupingAgg (5)
                                       +- PhotonProject (4)
                                          +- PhotonScan parquet  (3)


(1) PhotonScan parquet 
Output [4]: [event_id#13381,

| Aspect | Your Finding |
|---|---|
| How many times is events scanned? | 2 times — once for the first part and once for the aggregation subquery |
| Exchange count | 2 exchanges: 1 shuffle for aggregation on `track_id`, and 1 broadcast exchange for joining aggregated results back |
| Join strategy | `PhotonBroadcastHashJoin` on `track_id` |
| Could caching help? | Yes — caching the `events` dataset could avoid repeated scans when the same source is reused in aggregation + join workflows |
| Assessment | Efficient execution overall: aggregation result is small enough to broadcast back into the main query, minimizing join shuffle cost. However, the self-reference pattern causes the `events` table to be scanned twice, introducing redundant I/O that caching could reduce. |

# Part 3: Plan Analysis Report
# StreamPulse Execution Plan Audit Report
 
## Summary
- Total queries analyzed: `5`
- Queries needing optimization: `2` (Query 4 and Query 5)
- Most common issue: `Repeated scans and redundant joins/computation`
- Estimated total improvement potential: 15–35% reduction in repeated I/O and shuffle overhead through caching
 
## Query-by-Query Findings
 
### Query 1: Simple filter and select
- **Status:** Efficient
- **Issues found:** None
- **Recommendation:** Keep current design. Query already benefits from partition pruning, predicate pushdown, column pruning, and full Photon optimization.
 
### Query 2: Join events with artists
- **Status:** Efficient
- **Issues found:** Minor broadcast preparation exchange
- **Recommendation:** No major changes needed. Maintain small dimension tables to preserve broadcast hash join efficiency.
 
### Query 3: Three-table join
- **Status:** Efficient
- **Issues found:** Aggregation shuffle is unavoidable
- **Recommendation:** Keep current join strategy. 
 
### Query 4: Aggregation with multiple actions (simulated)
- **Status:** Needs Work
- **Issues found:** Same `events ↔ artists` enrichment executed twice independently
- **Recommendation:** Cache the enriched dataset before running multiple downstream aggregations.
 
### Query 5: Self-join pattern
- **Status:** Needs Work
- **Issues found:** events table scanned twice due to aggregation subquery + join-back pattern
- **Recommendation:** Cache projected event data or aggregation results to reduce repeated scans.
 
## Priority Recommendations
1. Cache reused enriched datasets (`events` joined with dimensions) to eliminate repeated joins and scans.
2. Reuse intermediate aggregation outputs for self-reference workflows instead of rescanning base tables.
3. Review shuffle partition count and tune based on cluster size and data volume.
 
## Configuration Recommendations
- `spark.sql.autoBroadcastJoinThreshold:` Keep enabled/increase if needed (broadcast joins are working effectively)
- `spark.sql.shuffle.partitions:` Tune dynamically based on workload size; current values appear reasonable for moderate datasets
- `Caching strategy:` Cache frequently reused enriched DataFrames or intermediate aggregates used across multiple queries

In [0]:
#Part 4: Optimization Proposals
"""q1_optimized = (events.filter((col("year") == 2024) & (col("completed") == True)).select("event_id", "user_id", "duration_sec"))"""

"""q2_optimized = events \
    .filter(col("year") == 2024) \
    .join(artists, "artist_id") \
    .select("event_id", "name", "genre", "duration_sec")"""

"""q3_optimized = (events.filter(col("year") == 2024)
    .join(tracks,"track_id")
    .join(artists,"artist_id")
    .filter(col("genre") == "Pop")
    .groupBy("name")
    .agg(count("*").alias("play_count"), avg("duration_sec").alias("avg_duration"))
    )
)"""

"""q4 optimized
enriched = events.filter(col("year") == 2024)
    .join(artists,"artist_id")
    .select("artist_id","genre","device","duration_sec")
    .cache()

# Materialize cache
enriched.count()

q4a = enriched.groupBy("genre").agg(count("*").alias("plays"))
q4a.explain(mode="formatted")
 
q4b = enriched.groupBy("device").agg(avg("duration_sec").alias("avg_dur"))
q4b.explain(mode="formatted")
)"""

"""q5 optimized

popular = events.groupBy("track_id").agg(
    .count("*").alias("play_count"))
    .filter(col("play_count") > 10)
    .cache()

# Materialize cache
popular.count()

q5 = events.join(popular, "track_id") \
    .select("event_id", "user_id", "track_id", "play_count")
"""




'q5 optimized\n\npopular = events.groupBy("track_id").agg(\n    .count("*").alias("play_count"))\n    .filter(col("play_count") > 10)\n    .cache()\n\n# Materialize cache\npopular.count()\n\nq5 = events.join(popular, "track_id")     .select("event_id", "user_id", "track_id", "play_count")\n'